<a href="https://colab.research.google.com/github/mbmanoj904-hue/Generative-ai/blob/main/JSON_File.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# AI EMAIL ASSISTANT - COMPLETE PROJECT IN ONE COLAB CELL
# Gemini + LangGraph + LangSmith + JSON
# ============================================================

# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

!pip install -q -U \
    langgraph \
    langchain \
    langchain-core \
    google-genai \
    langsmith


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import json
from typing import TypedDict

from google import genai
from google.colab import userdata

from langgraph.graph import StateGraph, START, END


# ============================================================
# 3. GET API KEYS FROM COLAB SECRETS
# ============================================================

GEMINI_API_KEY = (
   userdata.get("Geminiai")
)

LANGSMITH_API_KEY = userdata.get("Langsmith")


if not GEMINI_API_KEY:
    raise ValueError(
        "Gemini API key not found. "
        "Add GEMINI_API_KEY or Geminiai in Colab Secrets."
    )

if not LANGSMITH_API_KEY:
    raise ValueError(
        "LangSmith API key not found. "
        "Add LANGSMITH_API_KEY in Colab Secrets."
    )


# ============================================================
# 4. LANGSMITH CONFIGURATION
# ============================================================

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "AI-Email-Assistant"

print("LangSmith Project:", os.environ["LANGCHAIN_PROJECT"])


# ============================================================
# 5. GEMINI CLIENT
# ============================================================

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.8-flash"

print("Gemini Model:", MODEL_NAME)


# ============================================================
# 6. CREATE EMAIL JSON FILE
# ============================================================

email_data = [
    {
        "id": 1,
        "from": "hr@company.com",
        "subject": "Interview Invitation",
        "body": "You are invited to attend an interview for the Software Developer position.",
        "category": "Important",
        "priority": "High"
    },
    {
        "id": 2,
        "from": "manager@company.com",
        "subject": "Project Meeting Tomorrow",
        "body": "Please attend the project meeting tomorrow at 10 AM.",
        "category": "Important",
        "priority": "High"
    },
    {
        "id": 3,
        "from": "newsletter@example.com",
        "subject": "Weekly Newsletter",
        "body": "Here are this week's latest technology updates and articles.",
        "category": "Normal",
        "priority": "Low"
    },
    {
        "id": 4,
        "from": "offers@example.com",
        "subject": "Special Discount Offer",
        "body": "Get 50 percent discount on selected products. Offer ends soon.",
        "category": "Promotional",
        "priority": "Low"
    },
    {
        "id": 5,
        "from": "unknown@example.com",
        "subject": "You Won a Prize",
        "body": "Congratulations! You have won a large cash prize. Click the link to claim it.",
        "category": "Spam",
        "priority": "Low"
    }
]


# Save JSON file
with open("emails.json", "w") as f:
    json.dump(email_data, f, indent=4)

print("\nJSON file created: emails.json")


# ============================================================
# 7. LOAD JSON FILE
# ============================================================

with open("emails.json", "r") as f:
    emails = json.load(f)

print("Total emails:", len(emails))


# ============================================================
# 8. LANGGRAPH STATE
# ============================================================

class EmailState(TypedDict):

    email: dict

    classification: str
    summary: str
    priority: str
    reply: str

    final_result: str


# ============================================================
# 9. GEMINI FUNCTION
# ============================================================

def call_llm(prompt):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    if not response.text:
        raise ValueError("Empty response from Gemini")

    return response.text.strip()


# ============================================================
# 10. EMAIL CLASSIFIER
# ============================================================

def classify_email(state: EmailState):

    email = state["email"]

    prompt = f"""
You are an AI Email Classifier.

Analyze this email.

From:
{email["from"]}

Subject:
{email["subject"]}

Body:
{email["body"]}

Classify the email into exactly one category:

Important
Normal
Promotional
Spam

Return only the category.
"""

    try:

        result = call_llm(prompt)

        return {
            "classification": result
        }

    except Exception as e:

        return {
            "classification": "Unknown"
        }


# ============================================================
# 11. EMAIL SUMMARIZER
# ============================================================

def summarize_email(state: EmailState):

    email = state["email"]

    prompt = f"""
Summarize the following email in 1 or 2 simple sentences.

Subject:
{email["subject"]}

Body:
{email["body"]}

Return only the summary.
"""

    try:

        result = call_llm(prompt)

        return {
            "summary": result
        }

    except Exception:

        return {
            "summary": "Unable to summarize email."
        }


# ============================================================
# 12. PRIORITY DETECTOR
# ============================================================

def detect_priority(state: EmailState):

    email = state["email"]

    prompt = f"""
Determine the priority of this email.

Subject:
{email["subject"]}

Body:
{email["body"]}

Choose exactly one:

High
Medium
Low

Return only the priority.
"""

    try:

        result = call_llm(prompt)

        return {
            "priority": result
        }

    except Exception:

        return {
            "priority": "Medium"
        }


# ============================================================
# 13. REPLY GENERATOR
# ============================================================

def generate_reply(state: EmailState):

    email = state["email"]

    category = state["classification"]
    summary = state["summary"]
    priority = state["priority"]

    prompt = f"""
You are an AI Email Reply Assistant.

Generate a professional and polite reply to this email.

From:
{email["from"]}

Subject:
{email["subject"]}

Body:
{email["body"]}

Category:
{category}

Summary:
{summary}

Priority:
{priority}

Write a short professional reply.

Return only the reply.
"""

    try:

        result = call_llm(prompt)

        return {
            "reply": result
        }

    except Exception:

        return {
            "reply": "Thank you for your email. I will review it and get back to you."
        }


# ============================================================
# 14. FINAL RESULT NODE
# ============================================================

def final_node(state: EmailState):

    result = f"""
EMAIL ANALYSIS
---------------

From:
{state["email"]["from"]}

Subject:
{state["email"]["subject"]}

Category:
{state["classification"]}

Priority:
{state["priority"]}

Summary:
{state["summary"]}

Suggested Reply:
{state["reply"]}
"""

    return {
        "final_result": result
    }


# ============================================================
# 15. BUILD LANGGRAPH
# ============================================================

workflow = StateGraph(EmailState)

workflow.add_node(
    "classifier",
    classify_email
)

workflow.add_node(
    "summarizer",
    summarize_email
)

workflow.add_node(
    "priority",
    detect_priority
)

workflow.add_node(
    "reply",
    generate_reply
)

workflow.add_node(
    "final",
    final_node
)


# ============================================================
# 16. GRAPH FLOW
# ============================================================

workflow.add_edge(
    START,
    "classifier"
)

workflow.add_edge(
    "classifier",
    "summarizer"
)

workflow.add_edge(
    "summarizer",
    "priority"
)

workflow.add_edge(
    "priority",
    "reply"
)

workflow.add_edge(
    "reply",
    "final"
)

workflow.add_edge(
    "final",
    END
)


# ============================================================
# 17. COMPILE
# ============================================================

app = workflow.compile()

print("\nLangGraph compiled successfully!")


# ============================================================
# 18. SELECT EMAIL
# ============================================================

print("\nAvailable Emails:")

for email in emails:

    print(
        f'{email["id"]}. '
        f'{email["subject"]} '
        f'({email["from"]})'
    )


email_id = int(
    input("\nEnter email ID to analyze: ")
)


# Find selected email
selected_email = None

for email in emails:

    if email["id"] == email_id:

        selected_email = email
        break


if selected_email is None:

    raise ValueError("Invalid email ID")


# ============================================================
# 19. INITIAL STATE
# ============================================================

initial_state = {

    "email": selected_email,

    "classification": "",
    "summary": "",
    "priority": "",
    "reply": "",

    "final_result": ""
}


# ============================================================
# 20. RUN LANGGRAPH
# ============================================================

print("\nProcessing email...")

result = app.invoke(
    initial_state
)


# ============================================================
# 21. DISPLAY FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("AI EMAIL ASSISTANT RESULT")
print("=" * 70)

print(
    result["final_result"]
)

print("=" * 70)

print("\nLangSmith Project:")
print("AI-Email-Assistant")

print("\nOpen LangSmith to view the complete LLM trace.")

LangSmith Project: AI-Email-Assistant
Gemini Model: gemini-3.8-flash

JSON file created: emails.json
Total emails: 5

LangGraph compiled successfully!

Available Emails:
1. Interview Invitation (hr@company.com)
2. Project Meeting Tomorrow (manager@company.com)
3. Weekly Newsletter (newsletter@example.com)
4. Special Discount Offer (offers@example.com)
5. You Won a Prize (unknown@example.com)

Enter email ID to analyze: 1

Processing email...


AI EMAIL ASSISTANT RESULT

EMAIL ANALYSIS
---------------

From:
hr@company.com

Subject:
Interview Invitation

Category:
Unknown

Priority:
Medium

Summary:
Unable to summarize email.

Suggested Reply:
Thank you for your email. I will review it and get back to you.


LangSmith Project:
AI-Email-Assistant

Open LangSmith to view the complete LLM trace.
